In [47]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "manrique2011spontaneous")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "toolmakingthreespecies.sav")
complete_path_3 = os.path.join(original_data_pathway, "tool_selection_based_on_perceptual_properties_results.csv")
complete_path_4 = os.path.join(original_data_pathway, "selection_based_on_function_results.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [48]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_spss(complete_path_1, usecols=None, convert_categoricals=True)
# original_data_pathway_out = os.path.join(original_data_pathway, 'toolmakingthreespecies.csv')
# df.to_csv(original_data_pathway_out, encoding='utf-8-sig', index=False)


In [49]:
df['study_id']="manrique2011spontaneous"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)

df.rename(columns={"species": "species_original",
    "subject":"participant"}, inplace=True)

df['date']= pd.to_datetime(df['date'],format='%Y-%m-%d')
df['month']= df['date'].dt.month
df['day']= df['date'].dt.day
df['year']= df['date'].dt.year

experiment_list = [['cable', '1'],
                    ['twolidstube','2'],
                    ['metalbar','2']]
for x,y in experiment_list:
    df.loc[df.tooltype == x, ['experiment']] = y

session_reduction = [[16, 1],[17,2], [18,3], [19, 4], [20,5], [21, 6]]
for x,y, in session_reduction:
    df['session'].replace(x, y, inplace=True, regex=True)

In [50]:
df3 = pd.read_csv(complete_path_3)
df4 = pd.read_csv(complete_path_4)

new_datasets = [[df3, '3'], [df4, '4']]

for x,y in new_datasets:
    x[['day','month', 'year']] = x['Date'].str.split('/',expand=True)
    x['year'] = '20' + x['year'].astype(str)
    x['study_id']="manrique2011spontaneous"
    x['experiment']=y
    x.rename(columns={"Trial": "trial_temp"}, inplace=True)


In [51]:
out_df3_session = [i+1 for _ in range(7) for i in range(2) for _ in range(4)] 
out_df3_trial = [(i+1) for l in range(14) for i in range(4)]
out_df3_experience = [i+1 for _ in range(28) for i in range(2) for _ in range(1)] 
df3 = df3.assign(session=out_df3_session)
df3 = df3.assign(trial=out_df3_trial)
df3 = df3.assign(tool_set_experience=out_df3_experience)

In [52]:
out_df4_session = [i+1 for _ in range(6) for i in range(2) for _ in range(6)] 
out_df4_trial = [(i+1) for l in range(12) for i in range(6)]
df4 = df4.assign(session=out_df4_session)
df4 = df4.assign(trial=out_df4_trial)

In [53]:
df4['Tool_set'].unique()
spe_2=[]
spe_3=[] 
for index, row in df4.iterrows():
    if row['Tool_set']=='Warm up':
        spe_2.append("warm up")
        spe_3.append("")
    else:
        spe_3.append(row['Tool_set'])
        spe_2.append("test")
df4 = df4.assign(trial_type=spe_2)
df4 = df4.assign(demonstration=spe_3)

remove_demo = ['Set 1 ', 'Set 2 ', 'Set 3 ', 'Set 4 ']
for x in remove_demo:
    df4['demonstration'].replace(x, '', inplace=True, regex=True)

remove_set = [' drinking', ' Bubbles', ' Kellogs', ' Stereofoam blow']
for x in remove_set:
    df4['Tool_set'].replace(x, '', inplace=True, regex=True)
# df4['Tool_set'].unique()

In [54]:
data_frames=[df, df3, df4]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    data_frames[index]=x
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

In [55]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['participant'] = fulldf['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')

fulldf.dropna(subset=['participant'], inplace=True)

In [56]:
# fulldf.columns


fulldf['tooltype'].replace('twolidstube', 'dowel', inplace=True)
fulldf['year'].replace(2019, 2009, inplace=True)

fulldf['try'].replace(', ', '-', inplace=True, regex=True)
space_list = ['success','modif_cab','modif_lid','demonstration',
              'trial_type','tool_set', 'try']
for x in space_list:
    fulldf[x].replace(' ', '_', inplace=True, regex=True)
# fulldf.columns

fulldf['day']=fulldf['day'].astype(int)
fulldf['year']=fulldf['year'].astype(int)
fulldf['month']=fulldf['month'].astype(int)
# fulldf['day'].unique()

In [57]:


comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

In [58]:
fulldf=fulldf[['study_id', 'experiment','year', 'month', 'day','participant', 'age_in_years','sex','species',
        'session', 'trial','trial_type', 'condition', 'tooltype','tool_set', 'tool_set_experience',
       'suitability', 'modify', 'getsjuice', 'technic', 'modifiable',
       'success', 'modif_cab', 'modif_lid', 'modif_bar',  'demonstration', 'correct', 'try']]

for index in range(1,5):
    exp = fulldf[fulldf['experiment'] == str(index)]
    exp = exp.dropna(axis=1, how='all')
    comp_out_path = os.path.join(out_pathway, 'manrique2011spontaneous_exp'+str(index)+'_standardized.csv')
    exp.to_csv(comp_out_path, encoding='utf-8-sig', index=False)
    names = exp.columns.tolist()
    exp_g = pd.DataFrame(names)
    exp_g = exp_g.rename(columns={0: "column_name"})
    exp_g["description"] = ""
    exp_g=exp_g[["column_name", "description"]]
    comp_out_path_glossary = os.path.join(out_pathway, 'manrique2011spontaneous_exp'+str(index)+'_glossary.csv')
    exp_g.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)